In [ ]:
import torch as nn
from modules import * 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SetTransformer(nn.Module):
    def __init__(self, dim_input, num_outputs, dim_output,
            num_inds=32, dim_hidden=256, num_heads=4, ln=False):
        super(SetTransformer, self).__init__()
        self.enc = nn.Sequential(
                ISAB(dim_input, dim_hidden, num_heads, num_inds, ln=ln),
                ISAB(dim_hidden, dim_hidden, num_heads, num_inds, ln=ln))
        self.dec = nn.Sequential(
                PMA(dim_hidden, num_heads, num_outputs, ln=ln),
                SAB(dim_hidden, dim_hidden, num_heads, ln=ln),
                SAB(dim_hidden, dim_hidden, num_heads, ln=ln),
                nn.Linear(dim_hidden, dim_output))

    def forward(self, X):
        return self.dec(self.enc(X))
    
if __name__ == "__main__":
    model = SetTransformer(dim_input=4, num_outputs=256, dim_output=1).to(device)
    v = torch.randn(2, 100, 3).to(device)  
    y = model(v)
    print("Input:", v.shape)   
    print("Output:", y.shape) 

In [ ]:
import os
from sklearn.model_selection import train_test_split
import random
import pandas as pd
import numpy as np
import torch 

input_path =  r"C:\Users\mhenr\Documents\Github\spectrometer-nn\holePunch\Training_Diplex_Cookie_Batch_500_500f_1\Training_Diplex_Cookie_Batch_500_500f_1\geometries"
input_files = [f for f in os.listdir(input_path) if f.endswith('.csv')]
output_path = r"C:\Users\mhenr\Documents\Github\spectrometer-nn\holePunch\Training_Diplex_Cookie_Batch_500_500f_1\Training_Diplex_Cookie_Batch_500_500f_1\QR_S_Data"
output_files = [f for f in os.listdir(output_path) if f.endswith('.csv')]


def add_to_data(input_path, output_path):
  for f in os.listdir(input_path):
      if f.endswith('.csv'):
          input_files.append(f)

  for f in os.listdir(output_path):
      if f.endswith('.csv'):
          output_files.append(f)


input_path =  r"C:\Users\mhenr\Documents\Github\spectrometer-nn\holePunch\Training_Diplex_Cookie_Batch_500_500f_2\Training_Diplex_Cookie_Batch_500_500f_2\geometries"
output_path = r"C:\Users\mhenr\Documents\Github\spectrometer-nn\holePunch\Training_Diplex_Cookie_Batch_500_500f_2\Training_Diplex_Cookie_Batch_500_500f_2\son_files\QR_S_Data"
add_to_data(input_path, output_path)

input_path2 = r"C:\Users\mhenr\Documents\Github\spectrometer-nn\holePunch\Training_Diplex_Cookie_Batch_500_500f_0\Training_Diplex_Cookie_Batch_500_500f_0\geometries"
output_path2 = r"C:\Users\mhenr\Documents\Github\spectrometer-nn\holePunch\Training_Diplex_Cookie_Batch_500_500f_0\Training_Diplex_Cookie_Batch_500_500f_0\son_files\QR_S_Data"


data_pairs = list(zip(input_files, output_files))


train_pairs, temp_pairs = train_test_split(
    data_pairs, test_size=0.20, random_state=42, shuffle=True
)

val_pairs, test_pairs = train_test_split(
    temp_pairs, test_size=0.50, random_state=42, shuffle=True
)

train_x, train_y = zip(*train_pairs)
val_x, val_y = zip(*val_pairs)
test_x, test_y = zip(*test_pairs)


def files_to_tensor(x_paths, y_paths, n=3):
    all_x = []
    all_y = []
    
    num_intervals = 2 ** n
    
    for x_p, y_p in zip(x_paths, y_paths):
        # --- Process Input (Geometry) ---
        # Each file becomes a 300-element vector (100 points * 3 features)
        x_data = pd.read_csv(os.path.join(input_path, x_p), header=None).T.values
        all_x.append(x_data.flatten())
        
        # --- Process Output (Frequency Sweep) ---
        df = pd.read_csv(os.path.join(output_path,y_p), sep=",", skiprows=1)
        # Clean non-numeric rows
        df = df[pd.to_numeric(df.iloc[:, 0], errors="coerce").notna()]
        
        freqs = df.iloc[:, 0].astype(float).to_numpy()
        data  = df.iloc[:, 1:33].astype(float).to_numpy() # 32 columns of data

        # Define sampling points across the frequency spectrum
        edges   = np.linspace(freqs.min(), freqs.max(), num_intervals + 1)
        centers = 0.5 * (edges[:-1] + edges[1:])

        # Find the index of the frequency closest to each center
        # 
        idxs = np.abs(freqs[:, None] - centers[None, :]).argmin(axis=0)
        
        # Select those specific rows and flatten
        y_data = data[idxs].flatten()
        all_y.append(y_data)
        
        # shuffled_order = np.random.permutation(100)
        # augmented_data= x_data[shuffled_order, :]
        # all_x.append(augmented_data.T.flatten())
        # all_y.append(y_data)

    
    return torch.tensor(np.array(all_x), dtype=torch.float32), \
           torch.tensor(np.array(all_y), dtype=torch.float32)

# Create your final tensors
train_X, train_Y = files_to_tensor(train_x, train_y)
val_X, val_Y = files_to_tensor(val_x, val_y)
test_X, test_Y = files_to_tensor(test_x, test_y)

print(f"Final Training Tensor Shape: {train_X.shape}") # Should be [1600, 300]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



train_ds = TensorDataset(train_X, train_Y)
val_ds = TensorDataset(val_X, val_Y)

train_loader= DataLoader(train_ds, batch_size = 32,shuffle = True )
val_loader = DataLoader(val_ds, batch_size=32)

# # === 1. Load data ===
# train_loader, val_loader, dataset = build_loaders(
#     data_root = "."  ,
#     n=1,#modify this one for higher dim output
#     batch_size=4,
#     test_ratio=0.1,  # 9:1 split
#     seed=42
# )

# === 2. Build model ===
model = SetTransformer(
    dim_input=4,       # (x, y, radius)
    num_outputs=256,   # matching your original RNN output
    dim_output=1,      # scalar output per seed
    dim_hidden=128,
    num_heads=4,
    num_inds=32
).to(device)


criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay= 1e-4)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)

best_val_loss = float('inf')
patience = 100
no_improve_epochs = 0
num_epochs = 1000

train_losses, val_losses = [], []

# === 2. Training loop ===
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        x = x.view(x.size(0), 100, 4)  # reshape: (batch, 300) → (batch, 100, 3)
        
        optimizer.zero_grad()
        out = model(x)          # out: (batch, 256, 1)
        out = out.squeeze(-1)   # → (batch, 256)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)

    # === Validation ===
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            
            x = x.view(x.size(0), 100, 4)  # reshape here too
            
            out = model(x)
            out = out.squeeze(-1)
            loss = criterion(out, y)
            val_loss += loss.item() * x.size(0)

    val_loss /= len(val_loader.dataset)
    val_losses.append(val_loss)

    scheduler.step()

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # === Early stopping ===
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve_epochs = 0
        torch.save(model.state_dict(), "set_trans_best_model.pt")
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= patience:
            print("Early stopping triggered.")
            break

print("Training done. Best validation loss:", best_val_loss)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

def visualize_sampled_data(output_file_path, input_file_path, n=3):
    # 1. Load the raw data
    df = pd.read_csv(output_file_path, sep=",", skiprows=1)
    df = df[pd.to_numeric(df.iloc[:, 0], errors="coerce").notna()]
    
    freqs_raw = df.iloc[:, 0].astype(float).to_numpy()
    data_raw  = df.iloc[:, 1:33].astype(float).to_numpy() 

    # 2. Your Sampling Logic
    num_intervals = 2 ** n
    edges   = np.linspace(freqs_raw.min(), freqs_raw.max(), num_intervals + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    
    # Finding nearest indices
    idxs = np.abs(freqs_raw[:, None] - centers[None, :]).argmin(axis=0)
    
    # This is what your model actually "sees" after flattening
    sampled_data_matrix = data_raw[idxs] # Shape: (8, 32)
    y_vector = sampled_data_matrix.flatten() # Shape: (256,)



    geom_df = pd.read_csv(input_file_path, header=None)
    print(geom_df.shape)

    x_data = geom_df.to_numpy(dtype=float).T   # shape: (N, 3)
    input_tensor = torch.tensor(x_data, dtype=torch.float32).unsqueeze(0).to(device)  # shape: (1, N, 3)


    model = SetTransformer(
        dim_input=3,       # (x, y, radius)
        num_outputs=256,   # matching your original RNN output
        dim_output=1,      # scalar output per seed
        dim_hidden=128,
        num_heads=4,
        num_inds=32
    ).to(device)

    #loading with gpu
    # model.load_state_dict(torch.load("set_trans_best_model.pt", weights_only=True))

    #loading with cpu
    model.load_state_dict(torch.load("set_trans_best_model.pt", weights_only=True, map_location=torch.device('cpu')))

    model.eval()

    with torch.no_grad():
            # Get prediction
            prediction = model(input_tensor)
            
            # Move to CPU and numpy
            y_pred_vector = prediction.cpu().numpy().flatten()


    # 3. Plotting Setup (4x4 grid for 16 S-parameters)
    s_params = ['S11', 'S12', 'S13', 'S14', 'S21', 'S22', 'S23', 'S24', 
                'S31', 'S32', 'S33', 'S34', 'S41', 'S42', 'S43', 'S44']
    
    fig, axes = plt.subplots(4, 4, figsize=(18, 14))
    axes = axes.flatten()
    
    for i, name in enumerate(s_params):
        # Column for Magnitude is 0, 2, 4... (2*i)
        col_idx = 2 * i
        
        # Extract the sampled points from the flattened vector (stride 32)
        # 
        sampled_curve = y_vector[col_idx::32]
        pred_curve  = y_pred_vector[col_idx::32]
        # Plot the Raw Simulation Data (High resolution)
        axes[i].plot(freqs_raw, data_raw[:, col_idx], color='lightgray', label='Raw Sim', alpha=0.5)
        
        # Plot the Sampled Points (What the NN learns)
        axes[i].scatter(centers, sampled_curve, color='blue', s=40, label='Sampled Points', zorder=3)
        axes[i].plot(centers, sampled_curve, '-', alpha=0.6)
        

        axes[i].plot(centers, pred_curve, color='olive', linestyle='--', marker='', label='Model Prediction', zorder=3)
        axes[i].set_title(f"{name} Magnitude")
        axes[i].grid(True, linestyle=':', alpha=0.7)
        if i == 0: axes[i].legend()

    plt.suptitle(f"Sampling Verification (n={n}, Total Points={len(y_vector)})", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Run it on one of your files
example_file = r"C:\Users\mhenr\Documents\Github\spectrometer-nn\holePunch\Training_Diplex_Cookie_Batch_500_500f_1\Training_Diplex_Cookie_Batch_500_500f_1\QR_S_Data\dip_cookie_500_0003.csv"
input_file = r"Training_Diplex_Cookie_Batch_500_500f_1\Training_Diplex_Cookie_Batch_500_500f_1\geometries\dip_cookie_geom_500_0003.csv"

visualize_sampled_data(example_file, input_file)

In [ ]:
geom_df = pd.read_csv(input_file, header=None)
print(geom_df.shape)
print(geom_df.head())